# NPTEL Study Notes: Knowledge Graph Completion and Evaluation

---

## 1. Introduction to Knowledge Graph Completion (KGC)

### 1.1 The Incompleteness Problem

Real-world Knowledge Graphs (KGs) such as Freebase, WordNet, and Wikidata are inherently **incomplete**. They suffer from the **Open-World Assumption (OWA)**:

* A triple missing from a Knowledge Graph is **not necessarily false**; it is simply unobserved or unknown.

The objective of **Knowledge Graph Completion (KGC)** (often cast as **link prediction**) is to infer missing fact triples with a calculated confidence level, outputting either:

1. **Hard decisions:** Classifying a triple strictly as valid (1) or invalid (0).
2. **Soft probabilistic judgments:** Assigning a continuous belief or probability score to the validity of the triple.

### 1.2 Information Modalities for Node/Edge Representations

To perform KGC, a machine learning system learns distributed embedding representations of entities and relations. There are two primary paradigms:

* **Topology-Only:** Embeddings are derived strictly from graph connectivity patterns and neighborhood structural topology.
* **Multimodal / Text-Augmented:** Structural graph topology is enriched with external information such as entity aliases, textual descriptions, surface names, and relations (widely leveraged when integrating KGs with Large Language Models).

---

## 2. Mathematical Formalism and the Scoring Function $f$

### 2.1 Notation

Let a Knowledge Graph be defined as $\mathcal{G} = (\mathcal{E}, \mathcal{R})$, where:

* $\mathcal{E}$ denotes the set of entities (cardinality $\vert{}\mathcal{E}\vert{} = E$).
* $\mathcal{R}$ denotes the set of relation types (cardinality $\vert{}\mathcal{R}\vert{} = R$).
* A fact is structured as a triple:

$$(s, r, o)$$



where $s \in \mathcal{E}$ is the **subject**, $r \in \mathcal{R}$ is the **relation**, and $o \in \mathcal{E}$ is the **object**.

Embeddings are denoted in boldface:

* $\mathbf{s} \in \mathbb{R}^d$ (or $\mathbb{C}^d$): Subject embedding
* $\mathbf{r} \in \mathbb{R}^d$ (or $\mathbb{C}^d$): Relation embedding
* $\mathbf{o} \in \mathbb{R}^d$ (or $\mathbb{C}^d$): Object embedding

> **Asymmetry Property:** Entities must be explicitly partitioned into subject ($s$) and object ($o$) roles because most real-world relations are **asymmetric** (e.g., $\text{CEO\_of}(s, o) \neq \text{CEO\_of}(o, s)$). Symmetrical relations (e.g., $\text{sibling\_of}$) represent a restricted subset.

### 2.2 The Scoring Function $f(s, r, o)$

The core of any KGC model is a scoring function:


$$f: \mathcal{E} \times \mathcal{R} \times \mathcal{E} \to \mathbb{R}$$

* $f(s, r, o)$ measures the **belief** or plausibility of the candidate triple.
* In neural architectures, $f$ acts as a parameterized function approximator.
* Geometric interpretations of $\mathbf{s}, \mathbf{r}, \mathbf{o}$ include:
* Points or displacement vectors in vector space.
* Hyperplane projections.
* Complex coordinate rotations.



### 2.3 Score Polarity Conventions

1. **Similarity / Plausibility Formulation:**

$$\text{Higher score } f(s, r, o) \implies \text{Higher confidence that triple is true}$$


2. **Distance-Based Energy Formulation (e.g., TransE):**

$$\text{Lower score/distance } f(s, r, o) \implies \text{Higher confidence that triple is true}$$



---

## 3. Training Paradigms & Loss Formulations

### 3.1 Softmax Probabilistic Formulation (Cross-Entropy Loss)

To map unbounded real-valued scores $f(s, r, o)$ into a normalized probability distribution over all entities:

#### Probability of Object Prediction:

$$P(o \mid s, r) = \frac{\exp(f(s, r, o))}{\sum_{o' \in \mathcal{E}} \exp(f(s, r, o'))}$$

#### Probability of Subject Prediction:

$$P(s \mid r, o) = \frac{\exp(f(s, r, o))}{\sum_{s' \in \mathcal{E}} \exp(f(s', r, o))}$$

#### Negative Log-Likelihood (NLL) Objective:

For all ground-truth triples $\mathcal{D}^+ \subset \mathcal{G}$, training minimizes the sum of negative log probabilities:


$$\mathcal{L}_{\text{NLL}} = -\sum_{(s, r, o) \in \mathcal{D}^+} \Big[ \log P(o \mid s, r) + \log P(s \mid r, o) \Big]$$

#### Computational Bottleneck:

Calculating the denominator requires summing across all $E = \vert{}\mathcal{E}\vert{}$ entities, which scales to $\mathcal{O}(\vert{}\mathcal{E}\vert{})$ per step and becomes computationally prohibitive for graphs where $\vert{}\mathcal{E}\vert{} \sim 10^5 - 10^7$.

---

### 3.2 Negative Sampling & The Local Closed-World Assumption (LCWA)

To avoid computing the full softmax denominator, models use **negative sampling**:

1. **Local Closed-World Assumption (LCWA):** Assumes unobserved facts generated by randomly perturbing either the subject or the object of an existing true triple are false.
2. Given a positive triple $(s, r, o) \in \mathcal{D}^+$:
* **Corrupted Object:** $(s, r, o') \quad \text{where } o' \in \mathcal{E} \setminus \{o\}$
* **Corrupted Subject:** $(s', r, o) \quad \text{where } s' \in \mathcal{E} \setminus \{s\}$


3. **Denominator Approximation:**
If sampling $K$ negative samples out of $E$ total entities uniformly at random:

$$\mathbb{E}\left[\sum_{o' \in \mathcal{E}} \exp(f(s, r, o'))\right] \approx \frac{E}{K} \sum_{i=1}^{K} \exp(f(s, r, o'_i))$$



*Note: If the gold entity is accidentally excluded during uniform sampling, explicitly restoring it into the sample set introduces finite-sample estimator bias.*

---

### 3.3 Margin-Based Ranking Loss (Hinge / ReLU Loss)

Instead of computing normalized probabilities, margin-based optimization forces positive facts to score higher than corrupted negative facts by at least an additive margin $\gamma > 0$.

#### Formulation (Higher Score = More True):

$$\mathcal{L}_{\text{hinge}} = \sum_{(s, r, o) \in \mathcal{D}^+} \sum_{(s', r', o') \in \mathcal{D}^-} \max\Big(0, \, \gamma + f(s', r', o') - f(s, r, o)\Big)$$

Where:

* $f(s, r, o)$ is the "good" (positive) score.
* $f(s', r', o')$ is the "bad" (corrupted negative) score.
* $\gamma$ is the margin hyperparameter.
* If $f(s, r, o) \ge f(s', r', o') + \gamma$, the term inside evaluates to $\le 0$, resulting in zero loss.
* If violated, gradients backpropagate through $f$, adjusting embeddings $\mathbf{s}, \mathbf{r}, \mathbf{o}$.

#### Complexity Note:

Total combinatorial fact space is $\vert{}\mathcal{E}\vert{}^2 \vert{}\mathcal{R}\vert{}$. Because pairing every positive fact with all negative facts yields an intractable number of constraints, stochastic minibatch training perturbs **only one element at a time** (either $s$ or $o$) for $K$ negative samples.

---

## 4. Evaluation Methodologies & Metrics

Model evaluation is performed via link prediction queries:


$$(s, r, ?) \quad \text{or} \quad (?, r, o)$$


The trained model scores all candidate entities $e \in \mathcal{E}$, sorts them in descending order of plausibility, and evaluates the rank positions of the ground-truth entities.

```
Query: (s, r, ?)
Score all e in E  --->  Sort Descending  --->  Rank List: [o_1, o_2, o_3, ..., o_E]

```

### 4.1 Evaluation Metrics

| Metric | Formula | Description & Characteristics |
| --- | --- | --- |
| **Hits@K** | $\frac{1}{\vert{}\mathcal{Q}\vert{}} \sum_{q \in \mathcal{Q}} \mathbb{I}(\text{Rank}_q \le K)$ | Proportion of queries where a correct entity appears within the top $K$ positions. Value ranges in $[0, 1]$. Optimal for multi-answer relations (one-to-many / many-to-many). |
| **Mean Reciprocal Rank (MRR)** | $\frac{1}{\vert{}\mathcal{Q}\vert{}} \sum_{q \in \mathcal{Q}} \frac{1}{\text{Rank}_q}$ | Average of inverse ranks of the first correct answer. Ranges in $(0, 1]$. Heavily penalizes rank drops near the top (e.g., dropping from Rank 1 to 2 halves the score from $1.0$ to $0.5$). |
| **Mean Average Precision (MAP)** | $\frac{1}{\vert{}\mathcal{Q}\vert{}} \sum_{q \in \mathcal{Q}} \text{AP}_q$ | Averages precision across all recall cutoff levels for queries with multiple true answers. |

---

### 4.2 Raw vs. Filtered Evaluation Setting

KGC benchmarks frequently feature **one-to-many** or **many-to-many** relations.

* *Example:* Query `(Barack Obama, attended_institution, ?)` has multiple valid entities in the graph (e.g., *Occidental College*, *Columbia University*, *Harvard Law School*).

```
Raw Candidate Ranking:
Rank 1: o_1 (False)
Rank 2: o_2 (True - known from Training Set)
Rank 3: o_3 (False)
Rank 4: o_4 (Target Gold Test Entity)

```

* **Raw Evaluation:**
The target test entity $o_4$ is assigned **Rank 4**, even though Rank 2 is occupied by an objectively true fact from the training set.
* **Filtered Evaluation:**
All known true triples present in train, validation, or test sets (other than the specific test target) are removed from the candidate list before computing the rank.
* Removing $o_2$ shifts the effective rank of $o_4$ from **Rank 4 to Rank 3**.


* **Impact:** Unfiltered (raw) evaluations artificially depress MRR and Hits@K metrics. Performance comparisons across papers require identical filtering protocols.

---

## 5. Standard Benchmark Datasets

| Dataset | Source Ontology | Entities ($\vert{}\mathcal{E}\vert{}$) | Relations ($\vert{}\mathcal{R}\vert{}$) | Key Issue & Modern Fix |
| --- | --- | --- | --- | --- |
| **WN18** | WordNet (Lexical synsets) | $\sim 40,943$ | 18 | **Test set leakage via inverse relations** (e.g., `hypernym` vs `hyponym`). Resolved by **WN18RR** (inverses removed). |
| **FB15K** | Freebase (General facts) | $\sim 14,951$ | 1,345 | **Test set leakage via inverse relations**. Resolved by **FB15K-237** (inverse relations removed, leaving 237 relation types). |
| **YAGO** | Wikipedia, WordNet, GeoNames | Variable subsets | Variable | Large-scale semantic ontology used for extended link prediction. |

---

## 6. Worked Example: Evaluation Metrics Computation

### Problem Statement:

For a test query $(s, r, ?)$, the ranked predictions returned by a model over 8 entity candidates are:


$$[o_1, o_2, o_3, o_4, o_5, o_6, o_7, o_8]$$


Ground-truth status:

* $o_2$: True entity (present in training set).
* $o_6, o_8$: Designated ground-truth answers for the test query.
* All other entities are false.

Compute:

1. **Raw MRR**
2. **Raw MAP**
3. **Filtered MRR** (removing training hit $o_2$)
4. **Filtered Hits@3**

---

### Step-by-Step Solution:

#### 1. Raw MRR:

The first relevant test target appears at rank 6 ($o_6$).


$$\text{Raw MRR} = \frac{1}{\text{Rank}_1} = \frac{1}{6} \approx 0.1667$$

#### 2. Raw MAP:

Test ground-truth hits are at positions 6 and 8.

* Precision at rank 6 ($P@6$):

$$P@6 = \frac{1 \text{ relevant target}}{6} = \frac{1}{6}$$


* Precision at rank 8 ($P@8$):

$$P@8 = \frac{2 \text{ relevant targets}}{8} = \frac{2}{8} = \frac{1}{4}$$


* Average Precision:

$$\text{AP} = \frac{\frac{1}{6} + \frac{2}{8}}{2} = \frac{\frac{4}{24} + \frac{6}{24}}{2} = \frac{10}{48} = \frac{5}{24} \approx 0.2083$$



#### 3. Filtered MRR:

Removing training entity $o_2$ from the list shifts all subsequent positions up by 1:


$$\text{New list: } [o_1, o_3, o_4, o_5, o_6, o_7, o_8]$$

* New position of $o_6$: Rank 5
* New position of $o_8$: Rank 7

$$\text{Filtered MRR} = \frac{1}{5} = 0.2000$$



#### 4. Filtered Hits@3:

Checking the top 3 ranks of the filtered list: $[o_1, o_3, o_4]$.
Neither $o_6$ nor $o_8$ falls within the top 3.


$$\text{Filtered Hits@3} = 0.0$$

---

## 7. Practice Questions & Conceptual Checks

### Q1. (NPTEL Pattern MCQ)

**Under the Local Closed-World Assumption (LCWA) in KGC training, how are negative triples typically generated?**

* (A) By querying an external search engine for disproof.
* (B) By taking triples explicitly verified as false by human annotators.
* (C) By randomly replacing either the subject or the object of a known positive triple with an arbitrary entity from $\mathcal{E}$.
* (D) By reversing the direction of all asymmetric relations.

**Answer:** **(C)**

**Explanation:** LCWA assumes the graph is locally complete. Corrupting either the subject ($s'$) or object ($o'$) in a known true triple $(s, r, o)$ generates candidate negative triples without requiring explicit negative labeling in the dataset.

---

### Q2. (NPTEL Pattern Conceptual)

**Why does Mean Reciprocal Rank (MRR) apply a harsher penalty to changes in top-ranked items than to items ranked lower down?**

**Answer & Explanation:**

The reciprocal rank function $g(r) = \frac{1}{r}$ is convex and non-linear:


$$\vert{}g(1) - g(2)\vert{} = 1.0 - 0.5 = 0.5$$

$$\vert{}g(2) - g(\infty)\vert{} = 0.5 - 0.0 = 0.5$$


Dropping an entity from rank 1 to rank 2 incurs the same metric loss ($0.5$) as dropping an entity from rank 2 completely out of the retrieved candidates.

---

### Q3. (NPTEL Pattern Numerical)

**A model uses a margin-based hinge loss $\mathcal{L} = \max(0, \gamma + f(s', r, o') - f(s, r, o))$. Given $\gamma = 1.5$, positive score $f(s, r, o) = 3.2$, and corrupted negative score $f(s', r, o') = 2.4$, calculate the resulting loss.**

**Step-by-Step Calculation:**

1. Substitute values into the loss formula:

$$\mathcal{L} = \max(0, \, 1.5 + 2.4 - 3.2)$$


2. Compute the interior sum:

$$1.5 + 2.4 - 3.2 = 3.9 - 3.2 = 0.7$$


3. Apply the maximum operator:

$$\mathcal{L} = \max(0, 0.7) = 0.7$$



**Result:** The loss is **0.7**. Backpropagation will trigger because the positive triple failed to beat the negative triple by the full margin $\gamma = 1.5$.

---

## 8. Summary Sheet for Revision

* **Knowledge Graph Incompleteness:** Handled under the Open-World Assumption (missing $\neq$ false) and trained using the Local Closed-World Assumption (random entity replacement yields negative facts).
* **Core Goal:** Learn a scoring function $f(s, r, o)$ using continuous vector/geometric embeddings $\mathbf{s}, \mathbf{r}, \mathbf{o}$.
* **Loss Functions:**
* **Negative Log-Likelihood (Softmax):** Normalizes over all entities $\mathcal{E}$; uses negative sampling with scale correction factor $\frac{\vert{}\mathcal{E}\vert{}}{K}$ to stay tractable.
* **Hinge / Margin Loss:** $\max(0, \gamma + f_{\text{bad}} - f_{\text{good}})$. Optimizes separation between positive and corrupted negative facts.


* **Evaluation Standards:**
* Evaluated via query blanks: $(s, r, ?)$ and $(?, r, o)$.
* **Hits@K:** Measures presence in top $K$; suitable for one-to-many/many-to-many relations.
* **MRR:** Average inverse rank of the first hit; sensitive to shifts among top ranks.
* **Filtered Setting:** Eliminates known training/validation ground truths from candidate lists so models are not penalized for predicting valid alternative answers.


* **Key Benchmarks:** WN18 and FB15K (leakage-prone via inverse relations) $\to$ Replaced by **WN18RR** and **FB15K-237**.